# **Fase 3b · Medição**
### *Calcula a taxa de erro de palavra (WER), classifica os termos críticos e as palavras apostadas e monta a lista de trechos para a escuta dirigida, conforme `docs/fase3/01_desenho_experimental.md`.*

(Este caderno não usa os áudios nem a GPU. Lê apenas os JSON produzidos pelo caderno 3a e as transcrições de referência da fase 1, de modo que qualquer pessoa com acesso ao repositório pode reproduzir todas as medidas)

## **1. Preparação**

*Instala a biblioteca jiwer, que alinha dois textos palavra por palavra e conta as diferenças, conecta o Drive e define as pastas. As transcrições de referência são as mesmas versionadas em `docs/fase1/transcricoes/`, copiadas sem alteração para a pasta `referencias` do Drive. As tabelas produzidas são salvas em `resultados/fase3/medidas`.*

In [1]:
!pip install -q jiwer==4.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json
import pandas as pd
import jiwer

PASTA_PROJETO = '/content/drive/MyDrive/escuta-cerrada-poc'
PASTA_RESULTADOS = os.path.join(PASTA_PROJETO, 'resultados', 'fase3')
PASTA_REFERENCIAS = os.path.join(PASTA_PROJETO, 'referencias')
PASTA_MEDIDAS = os.path.join(PASTA_RESULTADOS, 'medidas')
os.makedirs(PASTA_MEDIDAS, exist_ok=True)

print(sorted(os.listdir(PASTA_REFERENCIAS)))
print(len([a for a in os.listdir(PASTA_RESULTADOS) if a.endswith('.json')]), 'arquivos JSON')

Mounted at /content/drive
['ref_faixa01.txt', 'ref_faixa02.txt', 'ref_faixa03.txt', 'ref_faixa04.txt', 'ref_faixa05.txt']
21 arquivos JSON


## **2. Regras fixadas no desenho experimental**

*Transcreve para o código as decisões da seção 5 e da seção 6 do desenho experimental. `USOS_INTERROGACAO` registra, faixa por faixa e na ordem em que aparecem, se cada `[?]` marca dúvida sobre a palavra anterior ou palavra perdida, conforme a tabela formulada. `CORINGA` é a palavra que ocupa o lugar de cada `[?]` e que aceita qualquer palavra da máquina sem contar como erro.*

In [ ]:
CORINGA = '__coringa__'

# Uso de cada [?] por faixa, na ordem em que aparecem (desenho experimental, seção 5)
USOS_INTERROGACAO = {
    1: ['perdida', 'duvida', 'perdida', 'duvida'],
    2: ['perdida', 'perdida', 'perdida', 'perdida'],
    3: ['duvida', 'duvida', 'duvida', 'duvida'],
    4: ['perdida'],
    5: ['duvida', 'duvida', 'duvida', 'duvida', 'perdida', 'perdida', 'perdida'],
}

TERMOS_CRITICOS = {
    1: ['pacaembu', 'dia do trabalho', 'sindicalização'],
    2: ['madrugada', 'meio-dia', 'tardezinha'],
    3: ['prostituição', 'mal necessário'],
    4: ['batucada', 'surdo', 'tamborim'],
    5: ['dom aires correia', 'batel', 'portugal', 'cabral', 'vera cruz'],
}

PALAVRAS_APOSTADAS = {
    2: ['coxilha', 'cinamomos', 'guaipeca', 'cocoricoca', 'treval'],
    5: ['dom aires correia', 'batel'],
}

## **3. Normalização**

*Antes da comparação, referência e saída da máquina passam pelas mesmas regras. A função `limpar` põe tudo em minúsculas, retira a pontuação (inclusive os travessões de início de fala da faixa 05) e preserva o hífen interno das palavras ("torna-se", "meio-dia"). Acentos e grafia ficam como estão, e números não são convertidos.*

*`preparar_referencia` aplica à transcrição humana as regras das marcações. Retira `{som: …}` e `[...]`, substitui cada `[?]` pelo coringa (levando junto a palavra duvidosa, quando é o caso) e separa as palavras de `{fala: …}`, que ficam fora do cálculo principal e entram na medida complementar da seção 6.*

*`preparar_maquina` lê as palavras do JSON com seus tempos. O Whisper às vezes devolve um pronome enclítico como peça separada ("-se", "-nos"), e a função o reúne à palavra anterior, para que "torna-se" seja comparado com "torna-se".*

In [ ]:
def limpar(texto):
    """Minúsculas, sem pontuação, hífen interno preservado. Devolve lista de palavras."""
    t = texto.lower()
    t = re.sub(r'[^\w\s-]', ' ', t)          # tudo o que não é letra, algarismo, espaço ou hífen
    t = re.sub(r'(?<!\w)-|-(?!\w)', ' ', t)  # hífen solto (travessão, hífen final de palavra cortada)
    return t.split()


def preparar_referencia(texto, faixa):
    """Aplica as regras da seção 5 do desenho experimental à transcrição de referência.

    Devolve a lista de palavras e, para cada uma, se veio de {fala: ...}.
    As marcações [?] viram a palavra coringa.
    """
    t = re.sub(r'\{som:[^}]*\}', ' ', texto)
    t = t.replace('[...]', '')

    # [?] na ordem em que aparecem
    usos = iter(USOS_INTERROGACAO.get(faixa, []))
    def trocar(m):
        uso = next(usos)
        if uso == 'duvida':
            return f' {CORINGA} '
        return f'{m.group(1)} {CORINGA} '
    t = re.sub(r'(\S*?)\s*\[\?\]', trocar, t)

    palavras, de_fala = [], []
    for parte in re.split(r'(\{fala:[^}]*\})', t):
        if parte.startswith('{fala:'):
            conteudo = parte[len('{fala:'):-1]
            for p in limpar(conteudo):
                palavras.append(p); de_fala.append(True)
        else:
            for p in limpar(parte):
                palavras.append(p); de_fala.append(False)
    return palavras, de_fala


def preparar_maquina(registro):
    """Palavras da saída do modelo, cada uma com início e fim em segundos."""
    # O Whisper às vezes devolve um pronome enclítico como peça separada ("-se", "-nos"),
    # sem espaço inicial. Essas peças são reunidas à palavra anterior ("torna-se").
    pecas = []
    for seg in registro['segmentos']:
        for w in seg.get('words', []):
            if pecas and not w['word'].startswith(' '):
                texto, inicio, _ = pecas[-1]
                pecas[-1] = (texto + w['word'], inicio, w['end'])
            else:
                pecas.append((w['word'], w['start'], w['end']))
    palavras, tempos = [], []
    for texto, inicio, fim in pecas:
        for p in limpar(texto):
            palavras.append(p)
            tempos.append((inicio, fim))
    return palavras, tempos

### 3.1 Conferência da normalização

Mostra, para cada faixa, quantas palavras a referência tem depois da normalização, quantas vêm de `{fala: …}` e o contexto de cada coringa, para conferir que cada `[?]` foi tratado conforme a tabela do desenho experimental.

In [ ]:
REFERENCIAS = {}
for n in range(1, 6):
    with open(os.path.join(PASTA_REFERENCIAS, f'ref_faixa{n:02d}.txt'), encoding='utf-8') as f:
        REFERENCIAS[n] = preparar_referencia(f.read(), n)

for n, (palavras, de_fala) in REFERENCIAS.items():
    print(f'Faixa {n:02d} · {len(palavras) - sum(de_fala)} palavras · {sum(de_fala)} de {{fala}}')
    for i, p in enumerate(palavras):
        if p == CORINGA:
            print('   ', ' '.join(palavras[max(0, i - 2):i + 2]))

Faixa 01 · 396 palavras · 4 de {fala}
    iniciativas passe-me __coringa__ cerrar
    lei orgânica __coringa__ por
    lavoura de __coringa__ cultura
    por exemplo __coringa__ nas
Faixa 02 · 329 palavras · 0 de {fala}
    gado invadindo __coringa__ para
    para fugir __coringa__ derrama
    um cavalo __coringa__ casco
    porque teve __coringa__ debaixo
Faixa 03 · 659 palavras · 0 de {fala}
    polícia o __coringa__ mais
    explorando a __coringa__ agir
    cheio o __coringa__ carioca
    só aos __coringa__ interessava
Faixa 04 · 59 palavras · 0 de {fala}
    depois do __coringa__ de
Faixa 05 · 477 palavras · 0 de {fala}
    a rosmaninho __coringa__ do
    do moinho __coringa__ do
    trigal moças __coringa__ morenas
    iam a __coringa__ e
    ele é __coringa__ é
    monte as __coringa__ no
    a vos __coringa__ cruzeiro


**Resultado.** Os vinte coringas estão nas posições definidas pela tabela do desenho experimental. Nos casos de dúvida, a palavra duvidosa saiu junto com a marcação ("lei orgânica fugita [?]" virou `lei orgânica __coringa__`), e nos casos de palavra perdida, a palavra anterior foi preservada ("passe-me [?] cerrar" virou `passe-me __coringa__ cerrar`). As quatro palavras de `{fala: …}` da faixa 01 ficam identificadas para a medida complementar.

## **4. Alinhamento e taxa de erro de palavra**

*O cálculo do WER começa por um alinhamento. O jiwer procura a forma de transformar a referência na saída da máquina com o menor número de operações, e cada palavra recebe uma de quatro marcas. Acerto é a palavra igual nos dois textos. Substituição é uma palavra da referência trocada por outra. Omissão é uma palavra da referência que a máquina não escreveu. Inserção é uma palavra que a máquina escreveu sem correspondente na referência.*

*A taxa é a soma de substituições (S), omissões (D, de deletion) e inserções (I), dividida pelo número de palavras da referência (N), sem contar os coringas. Como as inserções entram no numerador, o WER pode passar de 100%.*

*Uma limitação do método fica registrada aqui. Quando a máquina escreve texto inventado no lugar de fala real, o alinhamento tende a contar cada palavra inventada como substituição, porque uma troca custa menos que uma omissão mais uma inserção. O WER não distingue, portanto, a palavra mal ouvida da palavra inventada.*

In [ ]:
def alinhar(ref, hip):
    """Alinha referência e saída (jiwer) e devolve uma operação por posição.

    estado_ref[i]: 'acerto', 'substituicao', 'omissao' ou 'coringa'
    estado_hip[j]: 'acerto', 'substituicao', 'insercao' ou 'coringa'
    """
    saida = jiwer.process_words(' '.join(ref), ' '.join(hip))
    estado_ref = [None] * len(ref)
    estado_hip = [None] * len(hip)
    for b in saida.alignments[0]:
        if b.type == 'equal':
            for i in range(b.ref_start_idx, b.ref_end_idx):
                estado_ref[i] = 'acerto'
            for j in range(b.hyp_start_idx, b.hyp_end_idx):
                estado_hip[j] = 'acerto'
        elif b.type == 'substitute':
            for i, j in zip(range(b.ref_start_idx, b.ref_end_idx),
                            range(b.hyp_start_idx, b.hyp_end_idx)):
                op = 'coringa' if ref[i] == CORINGA else 'substituicao'
                estado_ref[i] = op
                estado_hip[j] = op
        elif b.type == 'delete':
            for i in range(b.ref_start_idx, b.ref_end_idx):
                estado_ref[i] = 'coringa' if ref[i] == CORINGA else 'omissao'
        elif b.type == 'insert':
            for j in range(b.hyp_start_idx, b.hyp_end_idx):
                estado_hip[j] = 'insercao'
    return estado_ref, estado_hip


def contar(estado_ref, estado_hip):
    n = sum(1 for e in estado_ref if e != 'coringa')
    s = estado_ref.count('substituicao')
    d = estado_ref.count('omissao')
    i = estado_hip.count('insercao')
    return {'N': n, 'acertos': estado_ref.count('acerto'), 'S': s, 'D': d, 'I': i,
            'WER': (s + d + i) / n if n else float('nan')}

### 4.1 Teste com resposta conhecida

Antes de medir o corpus, a função é testada com casos cuja resposta se sabe de antemão. "O samba é bom" contra "o samba e bom" tem uma substituição em quatro palavras, e o WER deve dar 0,25. Com um coringa no lugar de "samba", qualquer palavra ali, ou nenhuma, deve dar WER zero, com três palavras de referência.

In [ ]:
print(contar(*alinhar(limpar('O samba é bom.'), limpar('o samba e bom'))))
print(contar(*alinhar(['o', CORINGA, 'é', 'bom'], limpar('o samba é bom'))))
print(contar(*alinhar(['o', CORINGA, 'é', 'bom'], limpar('o é bom'))))

{'N': 4, 'acertos': 3, 'S': 1, 'D': 0, 'I': 0, 'WER': 0.25}
{'N': 3, 'acertos': 3, 'S': 0, 'D': 0, 'I': 0, 'WER': 0.0}
{'N': 3, 'acertos': 3, 'S': 0, 'D': 0, 'I': 0, 'WER': 0.0}


**Resultado.** Os três casos dão a resposta esperada (0,25, 0 e 0). A troca de "é" por "e" conta como erro, porque o desenho mantém os acentos, enquanto a pontuação e a maiúscula são neutralizadas. O coringa aceita tanto uma palavra qualquer quanto a ausência de palavra, e não entra no total de palavras da referência. Os mesmos casos integram os testes automáticos da seção 9.

### 4.2 WER por faixa e por condição

In [ ]:
def carregar_execucao(n, condicao):
    with open(os.path.join(PASTA_RESULTADOS, f'faixa{n:02d}_{condicao}.json'), encoding='utf-8') as f:
        return json.load(f)

ALINHAMENTOS = {}
linhas = []
for n in range(1, 6):
    palavras, de_fala = REFERENCIAS[n]
    ref = [p for p, f in zip(palavras, de_fala) if not f]
    for c in 'ABCD':
        hip, tempos = preparar_maquina(carregar_execucao(n, c))
        estado_ref, estado_hip = alinhar(ref, hip)
        ALINHAMENTOS[(n, c)] = (ref, estado_ref, hip, estado_hip, tempos)
        linhas.append({'faixa': n, 'cond.': c, **contar(estado_ref, estado_hip)})

wer = pd.DataFrame(linhas)
wer.to_csv(os.path.join(PASTA_MEDIDAS, 'wer_por_faixa.csv'), index=False)
print(wer.round(3).to_string(index=False))

 faixa cond.   N  acertos   S   D  I   WER
     1     A 392      344  47   1 54 0.260
     1     B 392      131  59 202  1 0.668
     1     C 392      365  24   3 42 0.176
     1     D 392      346  40   6 35 0.207
     2     A 325      233  91   1 67 0.489
     2     B 325      262  58   5 21 0.258
     2     C 325      182 133  10 40 0.563
     2     D 325      205 104  16 32 0.468
     3     A 655      624  23   8 27 0.089
     3     B 655      592  61   2 74 0.209
     3     C 655      629  23   3 12 0.058
     3     D 655      575  66  14 46 0.192
     4     A  58       52   4   2  7 0.224
     4     B  58       51   5   2  6 0.224
     4     C  58       51   6   1  8 0.259
     4     D  58       49   7   2  5 0.241
     5     A 470      101  44 325  0 0.785
     5     B 470       20  41 409  0 0.957
     5     C 470      393  71   6 41 0.251
     5     D 470       20  41 409  0 0.957


### 4.3 Visão comparativa

A primeira tabela dispõe o WER de cada faixa nas quatro condições, lado a lado. A segunda calcula a taxa global de cada condição, somando os erros de todas as faixas e dividindo pela soma das palavras de referência, o que dá a cada faixa um peso proporcional à sua extensão.

In [ ]:
print(wer.pivot(index='faixa', columns='cond.', values='WER').round(3), '\n')

glob = wer.groupby('cond.')[['N', 'S', 'D', 'I']].sum()
glob['WER'] = (glob['S'] + glob['D'] + glob['I']) / glob['N']
glob.to_csv(os.path.join(PASTA_MEDIDAS, 'wer_global.csv'))
print(glob.round(3))

cond.      A      B      C      D
faixa                            
1      0.260  0.668  0.176  0.207
2      0.489  0.258  0.563  0.468
3      0.089  0.209  0.058  0.192
4      0.224  0.224  0.259  0.241
5      0.785  0.957  0.251  0.957 

          N    S    D    I    WER
cond.                            
A      1900  209  337  155  0.369
B      1900  224  620  102  0.498
C      1900  257   23  143  0.223
D      1900  258  447  118  0.433


**Resultado.** Na linha de base (A), o erro vai de 8,9% (faixa 03) a 78,5% (faixa 05). A decomposição da seção 4.2 distingue fenômenos opostos. Na faixa 05, nas condições A, B e D, há de 325 a 409 omissões e nenhuma inserção, porque a máquina deixou de escrever a fala durante o loop "A CIDADE NO BRASIL", enquanto nas faixas 01 e 02 (A) as inserções correspondem a texto produzido a mais.

O glossário piora a taxa global (de 36,9% para 49,8%), com a exceção da faixa 02, em que a taxa cai de 48,9% para 25,8%. O filtro produz a menor taxa global (22,3%), mas o ganho vem quase todo da faixa 05, em que ele evitou o loop. Numa verificação exploratória, não prevista no desenho, a taxa global sem a faixa 05 fica em 23,2% na condição A e em 21,3% na condição C. Na faixa 04, com 58 palavras de referência, as diferenças entre condições correspondem a uma ou duas palavras.

## **5. Termos críticos e palavras apostadas**

*Cada ocorrência de um termo na referência é classificada a partir do alinhamento. É mantida se todas as palavras registradas forem acertos, omitida se todas forem omissões e substituída nos demais casos. Conta como inventada cada ocorrência do termo na saída da máquina que não corresponda a um acerto, com o tempo em que aparece. Os termos são procurados na forma exata da lista, sem flexões, de modo que "meio dia", sem hífen, não equivale a "meio-dia".*

In [ ]:
def ocorrencias(palavras, termo):
    alvo = limpar(termo)
    k = len(alvo)
    return [list(range(i, i + k)) for i in range(len(palavras) - k + 1)
            if palavras[i:i + k] == alvo]


def classificar_termos(termos, ref, estado_ref, hip, estado_hip, tempos):
    linhas = []
    for termo in termos:
        for idx in ocorrencias(ref, termo):
            estados = [estado_ref[i] for i in idx]
            if all(e == 'acerto' for e in estados):
                classe = 'mantido'
            elif all(e == 'omissao' for e in estados):
                classe = 'omitido'
            else:
                classe = 'substituído'
            linhas.append({'termo': termo, 'classe': classe, 'inicio': None})
        for idx in ocorrencias(hip, termo):
            if not all(estado_hip[j] == 'acerto' for j in idx):
                linhas.append({'termo': termo, 'classe': 'inventado',
                               'inicio': tempos[idx[0]][0]})
    return linhas

In [ ]:
linhas = []
for (n, c), (ref, estado_ref, hip, estado_hip, tempos) in ALINHAMENTOS.items():
    for lista, termos in [('crítico', TERMOS_CRITICOS.get(n, [])),
                          ('apostada', PALAVRAS_APOSTADAS.get(n, []))]:
        for x in classificar_termos(termos, ref, estado_ref, hip, estado_hip, tempos):
            linhas.append({'lista': lista, 'faixa': n, 'cond.': c, **x})

termos = pd.DataFrame(linhas)
termos.to_csv(os.path.join(PASTA_MEDIDAS, 'termos.csv'), index=False)

tabela = (termos.groupby(['lista', 'faixa', 'termo', 'classe', 'cond.']).size()
          .unstack('cond.', fill_value=0))
print(tabela.to_string())

cond.                                         A  B  C  D
lista    faixa termo             classe                 
apostada 2     cinamomos         mantido      0  1  0  0
                                 substituído  1  0  1  1
               cocoricoca        mantido      1  1  0  0
                                 substituído  0  0  1  1
               coxilha           mantido      0  0  1  2
                                 substituído  2  2  1  0
               guaipeca          mantido      0  1  1  1
                                 substituído  1  0  0  0
               treval            substituído  1  1  1  1
         5     batel             omitido      1  1  0  1
                                 substituído  0  0  1  0
               dom aires correia omitido      1  1  0  1
                                 substituído  0  0  1  0
crítico  2     madrugada         mantido      2  2  2  2
               meio-dia          mantido      0  2  1  2
                               

**Resultado.** Os termos contextuais da faixa 01 (Pacaembu, Dia do Trabalho, sindicalização) e "Vera Cruz", ausentes da fala, não foram escritos em nenhuma condição. Na faixa 02, "meio-dia" foi contado como substituído na condição A porque a máquina escreveu "Meio dia", sem hífen, e mantido com o glossário, que trazia a grafia hifenizada. Das palavras apostadas, coxilha, cinamomos, guaipeca e treval foram trocadas na condição A, e na condição B cinamomos, cocoricoca e guaipeca foram mantidas, embora não estivessem no glossário. Na faixa 04, "batucada" se perdeu nas duas condições com filtro. Na faixa 05, todos os termos falados foram omitidos nas condições A, B e D, por causa do loop, e na condição C "Portugal" foi mantido e os demais foram trocados.

### 5.1 Termos inventados, com o tempo em que aparecem

In [ ]:
inv = termos[termos['classe'] == 'inventado'].copy()
inv['tempo'] = inv['inicio'].map(lambda s: f'{int(s // 60)}:{s % 60:04.1f}')
print(inv[['lista', 'faixa', 'cond.', 'termo', 'tempo']].to_string(index=False))

  lista  faixa cond.        termo  tempo
crítico      2     D   tardezinha 2:47.6
crítico      3     A prostituição 0:03.8
crítico      3     B prostituição 0:03.8
crítico      3     B prostituição 2:31.4
crítico      3     C prostituição 0:03.8
crítico      3     D prostituição 0:03.8
crítico      4     A     tamborim 1:00.7
crítico      4     C     tamborim 1:00.9


**Resultado.** A ocorrência de "prostituição" aos 3,8 segundos, nas quatro condições, decorre da marcação da referência ("prostitu[...]ção"), em que a retirada de `[...]` deixa a forma "prostitução", como confirmou a escuta dirigida. A segunda ocorrência da condição B (2:31) resulta da repetição de uma frase. As demais (faixa 02, condição D, e faixa 04, condições A e C) aparecem dentro de repetições do próprio modelo.

## **6. Medidas complementares**

*Duas medidas previstas no desenho experimental. As hesitações apagadas reincluem na referência as palavras marcadas com `{fala: …}` e mostram o que a máquina fez com cada uma. A grafia numérica lista as divergências que envolvem algarismos, para separar o erro de reconhecimento da simples diferença entre "500" e "quinhentos".*

In [ ]:
for n, (palavras, de_fala) in REFERENCIAS.items():
    if not any(de_fala):
        continue
    for c in 'ABCD':
        hip, _ = preparar_maquina(carregar_execucao(n, c))
        estado_ref, _ = alinhar(palavras, hip)
        print(f'Faixa {n:02d} {c}:', [(p, e) for p, e, f in zip(palavras, estado_ref, de_fala) if f])

Faixa 01 A: [('torna', 'omissao'), ('associ', 'omissao'), ('da', 'acerto'), ('industrial', 'omissao')]
Faixa 01 B: [('torna', 'substituicao'), ('associ', 'omissao'), ('da', 'acerto'), ('industrial', 'substituicao')]
Faixa 01 C: [('torna', 'omissao'), ('associ', 'omissao'), ('da', 'acerto'), ('industrial', 'omissao')]
Faixa 01 D: [('torna', 'substituicao'), ('associ', 'omissao'), ('da', 'acerto'), ('industrial', 'omissao')]


In [ ]:
for (n, c), (ref, estado_ref, hip, estado_hip, tempos) in ALINHAMENTOS.items():
    saida = jiwer.process_words(' '.join(ref), ' '.join(hip))
    for b in saida.alignments[0]:
        r = ref[b.ref_start_idx:b.ref_end_idx]
        h = hip[b.hyp_start_idx:b.hyp_end_idx]
        if b.type == 'substitute' and len(r) <= 3 and any(re.search(r'\d', w) for w in r + h):
            print(f'Faixa {n:02d} {c}:', ' '.join(r), '→', ' '.join(h))

Faixa 01 C: 33 → 36
Faixa 01 D: 33 → 36


**Resultado.** A máquina apagou as três hesitações da faixa 01 em todas as condições. A primeira é o "Torna..." interrompido por aplausos inesperados, rastro da interação entre o orador e a plateia, e a transcrição devolve o texto lido em vez da fala proferida. O acerto contado para "da" é um efeito do alinhamento, que atribuiu ao falso começo o "da" da frase corrigida. A única divergência com algarismos ("33" transcrito como "36", nas condições com filtro) é erro de reconhecimento, e não de notação.

## **7. Lista para a escuta dirigida**

*Reúne os trechos que atendem a pelo menos um dos critérios da seção 8 do desenho experimental. São sequências de três ou mais palavras inseridas pela máquina, segmentos com texto e `no_speech_prob` acima de 0,6 e termos inventados. Como as quatro condições se referem ao mesmo áudio, os trechos que se sobrepõem (com folga de dois segundos) são agrupados, para que cada passagem da gravação seja escutada uma única vez.*

*Os critérios não captam as invenções que o alinhamento contou como substituição, como o loop "A CIDADE NO BRASIL" da faixa 05, conforme a limitação registrada na seção 4.*

In [ ]:
def trechos_inseridos(hip, estado_hip, tempos, minimo=3):
    """Sequências de 'minimo' ou mais palavras inseridas pelo modelo."""
    trechos, atual = [], []
    for j, e in enumerate(estado_hip + [None]):
        if e == 'insercao':
            atual.append(j)
        else:
            if len(atual) >= minimo:
                trechos.append({'inicio': tempos[atual[0]][0], 'fim': tempos[atual[-1]][1],
                                'texto': ' '.join(hip[k] for k in atual)})
            atual = []
    return trechos

In [ ]:
itens = []
for (n, c), (ref, estado_ref, hip, estado_hip, tempos) in ALINHAMENTOS.items():
    for x in trechos_inseridos(hip, estado_hip, tempos):
        itens.append({'faixa': n, 'cond.': c, 'inicio': x['inicio'], 'fim': x['fim'],
                      'criterio': 'inserção', 'texto': x['texto']})
    for s in carregar_execucao(n, c)['segmentos']:
        if s['no_speech_prob'] > 0.6 and s['text'].strip():
            itens.append({'faixa': n, 'cond.': c, 'inicio': s['start'], 'fim': s['end'],
                          'criterio': 'sem fala?', 'texto': s['text'].strip()})
inv = termos[termos['classe'] == 'inventado']
for _, l in inv.iterrows():
    itens.append({'faixa': l['faixa'], 'cond.': l['cond.'], 'inicio': l['inicio'],
                  'fim': l['inicio'] + 1, 'criterio': 'termo inventado', 'texto': l['termo']})

itens = pd.DataFrame(itens).sort_values(['faixa', 'inicio']).reset_index(drop=True)

# agrupa por faixa os itens que se sobrepõem no tempo
trecho, fim_atual, faixa_atual = 0, None, None
numeros = []
for _, l in itens.iterrows():
    if l['faixa'] != faixa_atual or l['inicio'] > fim_atual + 2:
        trecho += 1
        faixa_atual, fim_atual = l['faixa'], l['fim']
    else:
        fim_atual = max(fim_atual, l['fim'])
    numeros.append(trecho)
itens['trecho'] = numeros

mmss = lambda s: f'{int(s // 60)}:{s % 60:04.1f}'
itens.to_csv(os.path.join(PASTA_MEDIDAS, 'escuta_dirigida.csv'), index=False)

for t, g in itens.groupby('trecho'):
    print(f"\nTrecho {t} · faixa {g['faixa'].iloc[0]:02d} · {mmss(g['inicio'].min())} a {mmss(g['fim'].max())}")
    for _, l in g.iterrows():
        print(f"   {l['cond.']} · {l['criterio']}: {l['texto'][:100]}")


Trecho 1 · faixa 01 · 1:25.6 a 1:29.8
   C · inserção: que seja que a gente não tenha mais um trabalho o que é que a gente tem que fazer
   A · inserção: exigência e desigualdade a questão é que a lei não é
   D · inserção: desigualdade e desigualdade

Trecho 2 · faixa 01 · 1:59.3 a 2:01.2
   D · inserção: a sua vida
   D · inserção: a sua vida como um ser humano o

Trecho 3 · faixa 01 · 2:06.4 a 2:14.8
   D · inserção: já há dieta e há conceito parawości de為什麼 memorable contra nossos coral grandes líderes

Trecho 4 · faixa 01 · 2:27.6 a 2:29.7
   A · inserção: mensagem do produção de uma mensagem de renovoção a
   A · inserção: destaca a lei e a lei como um sistema
   A · inserção: trabalho que em

Trecho 5 · faixa 01 · 2:55.3 a 2:55.9
   C · inserção: congreguem e que

Trecho 6 · faixa 01 · 2:58.8 a 2:59.6
   A · inserção: representam a lei e que se

Trecho 7 · faixa 01 · 3:29.4 a 3:29.6
   A · inserção: a reforma da lei orgânica a

Trecho 8 · faixa 01 · 5:24.7 a 5:35.3
   C · inser

**Resultado.** Os critérios selecionaram 31 trechos, escutados pela pesquisadora em 09/10/2026. Nenhum foi classificado como fala real ausente da referência. Vinte receberam apenas a categoria de fala real deformada, a maioria sem relação reconhecível com a fala, e os demais são repetições, texto sobre música, texto sobre ruído de superfície e um efeito da marcação da referência. O registro completo está em `docs/fase3/02_escuta_dirigida.md`.

## **8. Ambiente de execução**

In [4]:
import sys, importlib.metadata as md
from datetime import datetime
print(datetime.now().strftime('%d/%m/%Y %H:%M'))
print('Python', sys.version.split()[0])
for p in ['jiwer', 'pandas']:
    print(p, md.version(p))

13/10/2026 15:33
Python 3.13.16
jiwer 4.0.0
pandas 2.2.3


## **9. Testes automáticos**

*Executa os testes de `tests/test_wer.py`, que conferem, com casos de resposta conhecida, cada regra de normalização e de contagem do desenho experimental. As funções testadas estão em `src/medicao.py`, cópia idêntica das definidas neste caderno.*

In [3]:
!cd /content/drive/MyDrive/escuta-cerrada-poc && python -m pytest -q tests

.........                                                                [100%]
9 passed in 1.03s


**Resultado.** Os nove testes passaram (13/10/2026).